In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "fahrlytics"
spark.sql(f"USE CATALOG {CATALOG}")

# 1. Read Bronze raw trips
raw_trips = spark.table(f"{CATALOG}.bronze.trips")

# 2. Schema Transformation & Type Casts
transformed_trips = (
    raw_trips
    .withColumn("pickup_time", F.to_timestamp("pickup_time", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("drop_time", F.to_timestamp("drop_time", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("trip_date", F.to_date("pickup_time"))
    .withColumn("trip_duration_min", F.round((F.unix_timestamp("drop_time") - F.unix_timestamp("pickup_time")) / 60.0, 2))
    .withColumn("abs_distance_km", F.abs(F.col("distance_km")))
)

# 3. Add Operational Metrics & Quality Audit Flags
validated_trips = (
    transformed_trips
    .withColumn(
        "avg_speed_kmh",
        F.when(F.col("trip_duration_min") > 0, 
               F.round((F.col("abs_distance_km") / (F.col("trip_duration_min") / 60.0)), 2))
        .otherwise(0.0)
    )
    .withColumn("is_valid_distance", (F.col("distance_km") > 0.0) & (F.col("distance_km") <= 150.0))
    .withColumn("is_valid_duration", (F.col("trip_duration_min") >= 2.0) & (F.col("trip_duration_min") <= 240.0))
    .withColumn("is_valid_fare", F.col("fare_amount") >= 4.0)
    .withColumn("is_valid_speed", (F.col("avg_speed_kmh") > 0.0) & (F.col("avg_speed_kmh") <= 140.0))
    .withColumn("is_clean_record", 
        F.col("is_valid_distance") & F.col("is_valid_duration") & F.col("is_valid_fare") & F.col("is_valid_speed")
    )
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# 4. Deduplicate on trip_id using windowing (latest _ingested_at wins)
window_spec = Window.partitionBy("trip_id").orderBy(F.col("_ingested_at").desc(), F.col("pickup_time").desc())
deduped_trips = (
    validated_trips
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

# 5. Create Target Silver Table with Partitioning
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.silver.trips (
        trip_id             STRING,
        customer_id         STRING,
        driver_id           STRING,
        city                STRING,
        trip_date           DATE,
        pickup_time         TIMESTAMP,
        drop_time           TIMESTAMP,
        distance_km         DOUBLE,
        abs_distance_km     DOUBLE,
        fare_amount         DOUBLE,
        currency            STRING,
        surge_multiplier    DOUBLE,
        trip_status         STRING,
        vehicle_type        STRING,
        weather             STRING,
        is_ghost_trip       BOOLEAN,
        is_fare_inflated    BOOLEAN,
        day_type            STRING,
        trip_duration_min   DOUBLE,
        avg_speed_kmh       DOUBLE,
        is_valid_distance   BOOLEAN,
        is_valid_duration   BOOLEAN,
        is_valid_fare       BOOLEAN,
        is_valid_speed      BOOLEAN,
        is_clean_record     BOOLEAN,
        _ingested_at        TIMESTAMP,
        _silver_processed_at TIMESTAMP
    )
    USING DELTA
    PARTITIONED BY (city)
    COMMENT 'Silver conformed and validated trip transactions'
""")

# 6. Upsert into Silver via Delta MERGE INTO
target_table = DeltaTable.forName(spark, f"{CATALOG}.silver.trips")

(
    target_table.alias("target")
    .merge(
        deduped_trips.alias("source"),
        "target.trip_id = source.trip_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(f"Silver Trips merge complete. Total rows in silver.trips: {spark.table(f'{CATALOG}.silver.trips').count():,}")